In [2]:
%pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 115.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 60.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 145.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [3]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
max_seq_length=2048
dtype=None
load_in_4bit=True

In [5]:
model,tokenizer=FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct',
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit
)

==((====))==  Unsloth 2026.9.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [6]:
from peft.utils import loftq_utils
model=FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
            'gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.9.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [7]:
from datasets import load_dataset
dataset=load_dataset('ServiceNow-AI/R1-Distill-SFT','v0',split='train')

README.md:   0%|          | 0.00/2.70k [00:00<?, ?B/s]

v0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  180MB            

v0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

v0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  187MB            

v0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

v0/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  188MB            

v0/train-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/171647 [00:00<?, ? examples/s]

In [8]:
r1_prompt='''you are a reflective assistance engaging in thorough,iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt and continuous refinement before coming up wiht a answer
<problem>
{}
</problem>

{}
{}
'''
EOS_TOKEN=tokenizer.eos_token

def formatting_prompts_func(examples):
  problems=examples['problem']
  thoughts=examples['reannotated_assistant_content']
  solutions=examples['solution']
  texts=[]

  for problem,thought,solution in zip(problems,thoughts,solutions):
    text=r1_prompt.format(problem,thought,solution)+EOS_TOKEN
    texts.append(text)

  return{'text':texts}
dataset = dataset.map(formatting_prompts_func,batched=True,)


Map:   0%|          | 0/171647 [00:00<?, ? examples/s]

In [9]:
%pip install trl
%pip install -q transformers 

In [10]:
%pip install trl

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments,DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer=SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.1,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir='outputs',
        report_to='none',        
    ),
    
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/171647 [00:00<?, ? examples/s]

In [12]:
trainer_stats=trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 171,647 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.081573
2,1.002002
3,1.104595
4,1.025164
5,0.861588
6,0.932367
7,0.834895
8,0.815855
9,0.863989
10,0.808703


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [16]:
# 1. 使用与微调数据处理时一模一样的模板
r1_prompt = '''you are a reflective assistance engaging in thorough,iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt and continuous refinement before coming up wiht a answer
<problem>
{}
</problem>

{}
{}'''

# 2. 构建推理 Prompt，将 thought 和 solution 的位置留空，交由模型去补全
prompt_text = r1_prompt.format(
    "how many 'r' are present in the word 'strawberry'?", # 填入 problem
    "", # thought 留空
    ""  # solution 留空
)

# 3. 直接进行文本分词。这会默认返回一个包含 input_ids 和 attention_mask 的字典
inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")

# 4. 开启推理模式
FastLanguageModel.for_inference(model)

# 5. 使用 **inputs 解包字典，同时传入 input_ids 和 attention_mask 以消除警告
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    use_cache=True,
    temperature=1.5,
    min_p=0.1
)

# 6. 解码并强制在屏幕上打印出结果
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(response[0])

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


you are a reflective assistance engaging in thorough,iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt and continuous refinement before coming up wiht a answer
<problem>
how many 'r' are present in the word'strawberry'?
</problem>


<think>
Okay, I have this question: How many 'r' are present in the word "strawberry"? Hmm, alright. Let me try to figure this out.

First, I should read the word carefully: "strawberry". I know that words in English have letters in different positions, and sometimes these letters repeat, like in "dilly dally" where you have two "l"s. But in "strawberry", it's different. I remember that strawberries are fruits, so they're named after something, right?

Alright, I'm trying to remember if there's any pattern or rule in the English language when naming things that would make the letters in their names always or most frequently specific or similar in some way. Maybe the color, shape, or somet

In [17]:
# 1. 使用与微调数据处理时一模一样的模板
r1_prompt = '''you are a reflective assistance engaging in thorough,iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt and continuous refinement before coming up wiht a answer
<problem>
{}
</problem>

{}
{}'''

# 2. 构建推理 Prompt，将 thought 和 solution 的位置留空，交由模型去补全
prompt_text = r1_prompt.format(
    "how many bones are in the human body?", # 填入 problem
    "", # thought 留空
    ""  # solution 留空
)

# 3. 直接进行文本分词。这会默认返回一个包含 input_ids 和 attention_mask 的字典
inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")

# 4. 开启推理模式
FastLanguageModel.for_inference(model)

# 5. 使用 **inputs 解包字典，同时传入 input_ids 和 attention_mask 以消除警告
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    use_cache=True,
    temperature=1.5,
    min_p=0.1
)

# 6. 解码并强制在屏幕上打印出结果
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(response[0])

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


you are a reflective assistance engaging in thorough,iterative reasoning, mimicking human stream-of-consciousness thinking. Your approach emphasizes exploration, self-doubt and continuous refinement before coming up wiht a answer
<problem>
how many bones are in the human body?
</problem>


<think>
To determine the number of bones in the human body, I need to consider the skeleton system. I recall that the adult human body is composed of bones, but I need to make sure if all bones are counted correctly.

Let's start with the typical model of the human skeleton, which is composed of the skull, vertebral column, ribs, sternum, and appendicular skeleton (limbs). I know that each of these segments has its own bones. So, I can break it down into these categories.

First, the skull contains bones such as the cranium, face bones, and some additional bones at the back of the head. Similarly, the vertebral column consists of the cervical, thoracic, lumbar, sacral, and coccygeal regions, each wit